In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Shape:", X.shape)
print("Class counts:\n", y.value_counts())

Shape: (569, 30)
Class counts:
 1    357
0    212
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [4]:
X_unlab, X_lab, y_unlab, y_lab = train_test_split(X_train, y_train, test_size=0.05, random_state=42, stratify=y_train)

print("Labeled samples  :", len(X_lab))
print("Unlabeled samples:", len(X_unlab))

Labeled samples  : 20
Unlabeled samples: 378


In [5]:
model = LogisticRegression(max_iter=1000)
model.fit(X_lab, y_lab)

base_acc = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy with only labeled data: {base_acc:.4f}")

Accuracy with only labeled data: 0.9415


In [6]:
threshold = 0.95
X_l, y_l = X_lab.copy(), y_lab.copy()
X_u = X_unlab.copy()

for i in range(10):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_l, y_l)

    if len(X_u) == 0:
        break

    probs = model.predict_proba(X_u)
    confidence = probs.max(axis=1)
    pseudo_labels = probs.argmax(axis=1)

    confident = confidence >= threshold
    if confident.sum() == 0:
        print("No confident predictions left. Stopping.")
        break

    # add confident samples to labeled data
    X_l = pd.concat([X_l, X_u[confident]])
    y_l = pd.concat([y_l, pd.Series(pseudo_labels[confident], index=X_u[confident].index)])

    # remove them from unlabeled data
    X_u = X_u[~confident]

    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"Iteration {i+1}: added {confident.sum()} samples, labeled = {len(X_l)}, test accuracy = {acc:.4f}")

Iteration 1: added 236 samples, labeled = 256, test accuracy = 0.9415
Iteration 2: added 57 samples, labeled = 313, test accuracy = 0.9415
Iteration 3: added 10 samples, labeled = 323, test accuracy = 0.9474
Iteration 4: added 5 samples, labeled = 328, test accuracy = 0.9474
Iteration 5: added 3 samples, labeled = 331, test accuracy = 0.9474
Iteration 6: added 1 samples, labeled = 332, test accuracy = 0.9474
No confident predictions left. Stopping.


In [7]:
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_l, y_l)

self_acc = accuracy_score(y_test, final_model.predict(X_test))
print(f"Accuracy after self-training: {self_acc:.4f}")

Accuracy after self-training: 0.9474


In [8]:
full_model = LogisticRegression(max_iter=1000)
full_model.fit(X_train, y_train)

full_acc = accuracy_score(y_test, full_model.predict(X_test))
print(f"Accuracy with all labels: {full_acc:.4f}")

Accuracy with all labels: 0.9883


In [9]:
print(f"Labeled data only  : {base_acc:.4f}")
print(f"After self-training: {self_acc:.4f}")
print(f"All labels (limit) : {full_acc:.4f}")
print(f"Improvement        : {self_acc - base_acc:+.4f}")

Labeled data only  : 0.9415
After self-training: 0.9474
All labels (limit) : 0.9883
Improvement        : +0.0058
